# Job E: masked language models (PLL-word-l2r) on job D's substitutions, plus GPT-2-medium

**Run all cells.** Job E scores **the same substitutions as job D** with masked language models, which predict every
word from both sides. The score is PLL-word-l2r: each token is masked together with the rest of its word. The
queue cell (section 8) works through the items below in delivery order. Every item is resumable, and finished
items are skipped.

| tier | due | items |
|---|---|---|
| 1 | Sun 4 Oct | RoBERTa-base main |
| 2 | Tue 6 Oct | GPT-2-medium main, RoBERTa-large main |
| 3 | Thu 8 Oct | RoBERTa-base held-out, GPT-2-medium held-out |
| optional | if a GPU is free (`INCLUDE_OPTIONAL`) | RoBERTa-large held-out, ModernBERT-large main |

**Job D has priority.** Run this notebook on a GPU job D is not using. Everything runs in **float32 with TF32
off**, as the note requires.

Per item:
1. Load the model at the note's revision. Masked models load through the reference's own `load_mlm`; GPT-2-medium
   loads through job D's loader.
2. Size the batches automatically.
3. Run the **checks**:
   - tokenizer (the counts must equal the note's table exactly);
   - the engine against the unmodified reference;
   - the anchor;
   - **timing on the first 20 sentences**, with the projected hours printed and saved at once, so you can send
     them to the requesters.
4. Run the full set.
5. Check finite/complete, then write `tree_mlm_<model>_<set>.jsonl.gz` (or `tree_t4_gpt2-medium_<set>.jsonl.gz`)
   and `checks_<model>_<set>.json`. Both are pushed to Drive immediately.

## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

- `ITEMS`: `all` (the queue), or a comma list such as `roberta-base:main,gpt2-medium:main`.
- `INCLUDE_OPTIONAL`: add RoBERTa-large held-out and ModernBERT-large main.
- `GROUP_SUBS`: substituted sentences per packing pool for the masked models (each has ~20 masked copies).
- `PAUSE_AFTER_TIMING`: stop after each item's timing, for when you want the requesters' go-ahead first.

In [ ]:
#@title Run configuration { display-mode: "form" }
ITEMS              = "all"    #@param {type:"string"}
INCLUDE_OPTIONAL   = False    #@param {type:"boolean"}
LIMIT              = 0        #@param {type:"integer"}
GROUP_SUBS         = 4000     #@param {type:"integer"}
GROUP_SEQS_CAUSAL  = 20000    #@param {type:"integer"}
MIRROR_MINUTES     = 15       #@param {type:"number"}
STOP_ON_FAIL       = True     #@param {type:"boolean"}
PAUSE_AFTER_TIMING = False    #@param {type:"boolean"}
TOKENIZER_WORKERS  = 0        #@param {type:"integer"}
DRIVE_DIR          = "/content/drive/MyDrive/m1_llms_analyzer/tree_runner_job_e"  #@param {type:"string"}
OUTPUT_DIR         = "outputs/job_e"                                               #@param {type:"string"}
HF_CACHE_DIR       = ""       #@param {type:"string"}

import json
import shutil
from pathlib import Path

from m1_analyzer import in_colab
from m1_analyzer.experiments import job_d, job_e, tree_runner
from m1_analyzer.experiments import tree_runner_ref_v4 as ref

mref = job_e.mref
OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

ALL_ITEMS = [q for q in job_e.QUEUE_E if INCLUDE_OPTIONAL or q[2] != "optional"]
if ITEMS.strip() == "all":
    QUEUE = ALL_ITEMS
else:
    tiers = {(m, s): t for m, s, t in job_e.QUEUE_E}
    QUEUE = []
    for item in ITEMS.split(","):
        slug, set_name = item.strip().split(":")
        if (slug, set_name) not in tiers:
            raise SystemExit(f"{item!r} is not a job E item; choose from {sorted(tiers)}")
        QUEUE.append((slug, set_name, tiers[(slug, set_name)]))


def drive_dir():
    """The Drive folder, mounted on first use; None outside Colab."""
    if not in_colab():
        return None
    target = Path(DRIVE_DIR)
    if not Path("/content/drive").exists():
        from google.colab import drive  # noqa: F401 -- Colab only
        drive.mount("/content/drive")
    target.mkdir(parents=True, exist_ok=True)
    return target


def restore(path: Path) -> bool:
    """Copy `path` back from Drive when it is missing locally. True if it is now present."""
    if path.exists():
        return True
    drive = drive_dir()
    if drive is not None and (drive / path.name).exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(drive / path.name, path)
        print(f"restored {path.name} from Drive")
    return path.exists()


def mirror_path(path: Path):
    drive = drive_dir()
    return None if drive is None else drive / path.name


def push(path: Path) -> None:
    """Copy a finished file to Drive now."""
    if mirror_path(path):
        shutil.copy(path, mirror_path(path))
        print(f"  -> Drive: {mirror_path(path)}")


def save_json(path: Path, payload) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(payload, indent=1), encoding="utf-8")
    tmp.replace(path)
    push(path)


print(f"mlm reference : sha256 {job_e.mlm_reference_sha256()[:12]}…  unmodified={job_e.mlm_reference_unmodified()}")
print(f"tree reference: sha256 {job_d.reference_sha256()[:12]}…  unmodified={job_d.reference_unmodified()}")
if not (job_e.mlm_reference_unmodified() and job_d.reference_unmodified()):
    raise SystemExit("A reference file differs from the package's; restore it before running.")
SENTENCES = {s: job_d.load_sentences(set_name=s) for s in job_d.SETS}   # the same file as job D (sha256 checked)
ALL_SENTENCES = SENTENCES["main"] + SENTENCES["heldout"]
ANCHOR = job_e.load_anchor()
print(f"sentences     : main {len(SENTENCES['main'])}, heldout {len(SENTENCES['heldout'])}")
print(f"queue         : {[f'{m}:{s}' for m, s, _ in QUEUE]}")
print(f"drive         : {DRIVE_DIR if in_colab() else '(not in Colab)'}")

## 5 · Smoke test (no download, about a minute)

A random 2-layer RoBERTa masked LM with a small byte-level BPE tokenizer, trained here on the sentences. It checks
that the engine gives the unmodified `mlm_rows`'s values, that a resumed run adds nothing, and that the
deliverable is written. The numbers themselves mean nothing.

In [ ]:
#@title Smoke test on a tiny masked model
import time
import torch
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors, trainers
from transformers import PreTrainedTokenizerFast, RobertaConfig, RobertaForMaskedLM
import tempfile

started = time.perf_counter()
backend = Tokenizer(models.BPE())
backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
backend.decoder = decoders.ByteLevel()
backend.train_from_iterator([s["text"] for s in SENTENCES["main"][:300]], trainers.BpeTrainer(
    vocab_size=600, special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
backend.post_processor = processors.ByteLevel(trim_offsets=True)
stok = PreTrainedTokenizerFast(tokenizer_object=backend, bos_token="<s>", eos_token="</s>", cls_token="<s>",
                               sep_token="</s>", pad_token="<pad>", unk_token="<unk>", mask_token="<mask>")
torch.manual_seed(0)
smdl = RobertaForMaskedLM(RobertaConfig(vocab_size=len(stok), hidden_size=32, num_hidden_layers=2,
                                        num_attention_heads=2, intermediate_size=64, max_position_embeddings=200,
                                        pad_token_id=stok.pad_token_id)).eval()
sents = sorted(SENTENCES["main"][:20], key=lambda s: len(s["words"]))[:3]
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    budget = job_d.TokenBudget(3000)
    meta = {"model_id": "tiny-mlm", "revision": None, "dtype": "float32", "tf32": False}
    kw = dict(set_name="main", model_meta=meta, device="cpu", budget=budget, cache_path=tmp / "w.jsonl",
              show_progress=False, group_seqs=300)
    rows = job_e.run_item_mlm(stok, smdl, sents, **kw)
    assert job_e.run_item_mlm(stok, smdl, sents, **kw) == rows
    refr = [dict(id=s["id"], end=s["end"], **dict(zip(("orig", "rows"), mref.mlm_rows(stok, smdl, s["words"], s["keys"], s["end"]))))
            for s in sents]
    eq = job_e.check_equivalence_mlm(rows, refr)
    assert eq["pass"], eq
    m = job_e.item_meta_e(job_e.MODELS_E["roberta-base"], "main", meta, n_sentences=3, n_spans=0, wall_seconds=1.0,
                          checks={"engine_equivalence": eq, "complete": job_e.check_complete_mlm(rows, sents)}, budget=budget)
    job_d.finalize_item(rows, sents, tmp / "t.jsonl.gz", m)
print(f"engine vs mlm_rows: max |diff| total {eq['abs_diff']['total']['max']:.1e} nats on {eq['n_rows']} rows; "
      f"{time.perf_counter() - started:.1f}s")
print("Smoke test passed.")

## 6 · Tokenizer checks for every model (CPU only, no GPU needed)

- **Masked models:** `mlm_checks(tok, rows_of_that_set, causal_tok=...)`, with GPT-2's tokenizer for RoBERTa and
  none for ModernBERT. **Every count must equal the note's table exactly; if one differs, the cell stops.** Tell
  the requesters; don't work around it.
- **GPT-2-medium:** job D's `tokenizer_checks` and its gates.

The work is split over the CPU cores, and the results are cached per tokenizer and set: RoBERTa-base and -large
share their tokenizer files, so each set is computed once for both.

In [ ]:
#@title Run the tokenizer checks
import time
from transformers import AutoTokenizer

TOK_CHECKS = {}
_CAUSAL_TOK = {}


def causal_tok():
    if "gpt2" not in _CAUSAL_TOK:
        _CAUSAL_TOK["gpt2"] = job_e.load_causal_tok(cache_dir=HF_CACHE_DIR or None)
    return _CAUSAL_TOK["gpt2"]


def tokenizer_block(slug, set_name, tok=None):
    """The gated tokenizer block for one model and set, cached per tokenizer and set."""
    if (slug, set_name) in TOK_CHECKS:
        return TOK_CHECKS[(slug, set_name)]
    started = time.perf_counter()
    if job_e.is_causal(slug):
        spec = job_e.GPT2_MEDIUM
        if tok is None:
            tok, _ = job_d.load_tokenizer_d(spec, cache_dir=HF_CACHE_DIR or None)
        sha = job_d.tokenizer_sha(tok)
        path = OUT / job_d.tokenizer_check_name(sha, set_name)
        if restore(path):
            counts, source = json.loads(path.read_text())["counts"], "cached"
        else:
            counts = job_d.tokenizer_checks_parallel(tok, SENTENCES[set_name], workers=TOKENIZER_WORKERS or None)
            source = f"computed in {time.perf_counter() - started:.0f}s"
            save_json(path, {"tokenizer_sha256": sha, "set": set_name, "first_computed_for": spec.hf_id, "counts": counts})
        block = job_d.tokenizer_gate(counts, tok_sha=sha, model_id=spec.hf_id)
    else:
        spec = job_e.MODELS_E[slug]
        if tok is None:
            tok = AutoTokenizer.from_pretrained(spec.hf_id, revision=spec.revision, cache_dir=HF_CACHE_DIR or None)
        sha = job_d.tokenizer_sha(tok)
        path = OUT / f"mlmtok_{sha[:12]}_{'paired' if spec.causal_pair else 'alone'}_{set_name}.json"
        if restore(path):
            counts, source = json.loads(path.read_text())["counts"], "cached"
        else:
            counts = job_e.mlm_checks_parallel(tok, SENTENCES[set_name], causal_tok=causal_tok() if spec.causal_pair else None,
                                               workers=TOKENIZER_WORKERS or None)
            source = f"computed in {time.perf_counter() - started:.0f}s"
            save_json(path, {"tokenizer_sha256": sha, "set": set_name, "first_computed_for": spec.hf_id, "counts": counts})
        block = job_e.mlm_tokenizer_gate(counts, spec.family, set_name, tok_sha=sha, model_id=spec.hf_id)
    block = {**block, "set": set_name, "file": path.name}
    TOK_CHECKS[(slug, set_name)] = block
    shown = {k: v for k, v in block.items() if k in ("n_sentences", "n_substitutions", "n_forwards", "text_split_differs",
                                                    "frame_differs", "is_fast", "special_in_text", "roundtrip_fail",
                                                    "offsets_bad", "frame_pre_short", "frame_suf_short", "end_merge")}
    print(f"  {slug:<17} {set_name:<8} {'PASS' if block['pass'] else 'FAIL ' + str(block['failed'])}  ({source})  {shown}")
    return block


failed = []
for slug, set_name, _ in QUEUE:
    if not tokenizer_block(slug, set_name)["pass"]:
        failed.append(f"{slug}:{set_name}")
if failed:
    raise SystemExit(f"Tokenizer check FAILED for {failed}. Stop and send the tokenizer files to the requesters.")
print("\nAll tokenizer checks pass.")

## 7 · Status

In [ ]:
#@title What is done?
def status():
    print(f"{'item':<26}{'tier':>9}  {'due':<11}{'deliverable':<13}{'checks':<9}{'timing (h)':<11}work cache")
    for slug, set_name, tier in ALL_ITEMS:
        out, chk, work = (OUT / job_e.output_name(slug, set_name), OUT / job_e.checks_name(slug, set_name),
                          OUT / job_e.work_name(slug, set_name))
        drive = drive_dir()
        have = lambda p: p.exists() or (drive is not None and (drive / p.name).exists())
        c, h = "-", "-"
        if have(chk) and restore(chk):
            d = json.loads(chk.read_text())
            c = "PASS" if d.get("pass") else "FAIL"
            h = str(d.get("timing", {}).get("projected_hours", "-"))
        print(f"{slug + ':' + set_name:<26}{str(tier):>9}  {job_e.TIER_DUE_E[tier]:<11}{'DONE' if have(out) else '-':<13}"
              f"{c:<9}{h:<11}{'yes' if have(work) else '-'}")

status()

## 8 · Run the queue (GPU, resumable)

For every item in `QUEUE` that has no deliverable yet:
- **Model**: loaded once, then kept for the next item if it is the same model.
  - Masked models load through the reference's own `load_mlm`, in float32 with TF32 off.
  - GPT-2-medium loads through job D's loader, which checks the revision and the start token.
- **Auto-sizing**: calibrates for each model, as in job D. The masked probe uses the reference's `masked_logits`.
- **Checks** (stop on a blocking failure):
  - `tokenizer`: section 6.
  - `engine_equivalence`: 3 sentences (the shortest, a middle one and the longest of the first 50), scored by the
    unmodified `mlm_rows` (or `t4_rows`) and by the engine; every float within 0.01 nats.
  - `anchor`: `anchor_job_e.json`'s five sentences; integers identical, floats within 0.05 nats.
  - `timing`: the first 20 sentences (kept for the full run). The projected hours are printed and pushed to Drive in
    the checks file **before** the full run.
  - `causality`: GPT-2-medium only, reported.
- **Full run**, then `complete` (9 numbers per masked row, 8 for GPT-2-medium), then the deliverable.

In [ ]:
#@title Run every pending item
import gc, time
from datetime import datetime, timedelta, timezone

import torch

LOADED = {}


def ensure_model(slug):
    if LOADED.get("slug") == slug:
        return LOADED
    if LOADED:
        if LOADED.get("analyzer") is not None:
            LOADED["analyzer"].unload()
        LOADED.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    started = time.perf_counter()
    if job_e.is_causal(slug):
        tok, mdl, bos, device, analyzer, meta = job_d.load_model_d(job_e.GPT2_MEDIUM, cache_dir=HF_CACHE_DIR or None)
        budget, calib = job_d.calibrate_budget(mdl, bos, device, max_len=job_d.max_seq_len(tok, ALL_SENTENCES))
    else:
        spec = job_e.MODELS_E[slug]
        tok, mdl, device, meta = job_e.load_mlm_e(spec)
        bos, analyzer = None, None
        budget, calib = job_e.calibrate_budget_mlm(tok, mdl, device, max_len=job_d.max_seq_len(tok, ALL_SENTENCES))
    print(f"\nloaded {meta['model_id']} @ {meta['revision']}  float32  on {device} in {time.perf_counter() - started:.0f}s")
    print(f"auto-sizing: {calib}")
    LOADED.update(slug=slug, tok=tok, mdl=mdl, bos=bos, device=device, analyzer=analyzer, meta=meta, budget=budget,
                  calib=calib)
    return LOADED


def show(checks):
    for name, block in checks.items():
        if not isinstance(block, dict):
            continue
        state = "PASS" if block.get("pass") else ("FAIL" if "pass" in block else "reported")
        brief = {k: v for k, v in block.items() if k not in ("pass",) and not (isinstance(v, list) and not v)}
        print(f"  {name:<20} {state:<9} {json.dumps(brief)[:400]}")


def run_item(slug, set_name, L, work, sentences, **kw):
    common = dict(set_name=set_name, model_meta=L["meta"], device=L["device"], budget=L["budget"], cache_path=work,
                  mirror_path=mirror_path(work), mirror_minutes=MIRROR_MINUTES, **kw)
    if job_e.is_causal(slug):
        return job_d.run_item(L["tok"], L["mdl"], L["bos"], sentences, group_seqs=GROUP_SEQS_CAUSAL, **common)
    return job_e.run_item_mlm(L["tok"], L["mdl"], sentences, group_seqs=GROUP_SUBS, **common)


def score(slug, L, sentences):
    if job_e.is_causal(slug):
        return job_d.score_sentences(L["tok"], L["mdl"], L["bos"], sentences, device=L["device"], budget=L["budget"])
    return job_e.score_sentences_mlm(L["tok"], L["mdl"], sentences, device=L["device"], budget=L["budget"])


def run_checks(slug, set_name, L, work):
    causal = job_e.is_causal(slug)
    hf_id = job_e.GPT2_MEDIUM.hf_id if causal else job_e.MODELS_E[slug].hf_id
    tok, mdl, device = L["tok"], L["mdl"], L["device"]
    sents = SENTENCES[set_name]
    checks = {"tokenizer": tokenizer_block(slug, set_name)}
    if not checks["tokenizer"]["pass"]:
        return {**checks, "pass": False}
    first = sents[:50]
    by_len = sorted(first, key=lambda s: len(s["words"]))
    eq_sents = [by_len[0], by_len[len(by_len) // 2], by_len[-1]]
    t = time.perf_counter()
    ref_rows = []
    for s in eq_sents:
        if causal:
            orig, rows = ref.t4_rows(tok, mdl, L["bos"], s["words"], s["keys"], s["end"], batch=64, device=device)
        else:
            orig, rows = mref.mlm_rows(tok, mdl, s["words"], s["keys"], s["end"], batch=256, device=device)
        ref_rows.append({"id": s["id"], "end": s["end"], "orig": orig, "rows": rows})
    t_ref = time.perf_counter() - t
    t = time.perf_counter()
    fast = score(slug, L, eq_sents)
    eq = job_d.check_equivalence(fast, ref_rows, dtype="float32") if causal else job_e.check_equivalence_mlm(fast, ref_rows)
    checks["engine_equivalence"] = {**eq, "seconds_ref": round(t_ref, 1), "seconds_engine": round(time.perf_counter() - t, 1)}
    ids = {a["id"] for a in ANCHOR["models"][hf_id]["sentences"]}
    arows = score(slug, L, [s for s in ALL_SENTENCES if s["id"] in ids])
    checks["anchor"] = job_d.check_anchor(arows, ANCHOR, hf_id) if causal else job_e.check_anchor_mlm(arows, ANCHOR, hf_id)
    first20 = sents[:job_e.TIMING_FIRST]
    rows20 = run_item(slug, set_name, L, work, sents, ids=[s["id"] for s in first20], show_progress=True)
    checks["timing"] = job_e.timing_block(rows20, first20, sents, masked=not causal)
    if causal:
        checks["causality"] = job_d.check_causality(list(fast) + list(arows) + list(rows20), "float32")
    checks["pass"] = job_d.blocking_pass(checks)
    checks.update(token_budget=L["budget"].as_meta(), calibration=L["calib"], revision=L["meta"]["revision"],
                  reference_code_sha=job_d.REFERENCE_SHA256, mlm_reference_sha=job_e.MLM_REFERENCE_SHA256,
                  engine_version=job_d.ENGINE_VERSION if causal else job_e.ENGINE_VERSION)
    return checks


def retire_stale(slug, work: Path) -> None:
    """Rename a working cache written under another reference or engine (local and Drive)."""
    if not work.exists():
        return
    head = job_d.cache_header(work)
    want = job_d.ENGINE_VERSION if job_e.is_causal(slug) else job_e.ENGINE_VERSION
    ok = (head.get("reference_code_sha") == job_d.REFERENCE_SHA256 and head.get("engine") == want
          and (job_e.is_causal(slug) or head.get("mlm_reference_sha") == job_e.MLM_REFERENCE_SHA256))
    if ok:
        return
    tag = f".stale-{str(head.get('reference_code_sha'))[:8]}-{head.get('engine')}"
    work.rename(work.with_name(work.name + tag))
    if mirror_path(work) and mirror_path(work).exists():
        mirror_path(work).rename(mirror_path(work).with_name(work.name + tag))
    print(f"  {work.name} was written under another reference/engine; renamed to {work.name + tag}.")


def run_one(slug, set_name, tier):
    causal = job_e.is_causal(slug)
    sents = SENTENCES[set_name]
    out, chk, work = (OUT / job_e.output_name(slug, set_name), OUT / job_e.checks_name(slug, set_name),
                      OUT / job_e.work_name(slug, set_name))
    if restore(out):
        print(f"{slug}:{set_name} is done ({out.name}); skipping.")
        return
    print(f"\n=== {slug}:{set_name}  (tier {tier}, due {job_e.TIER_DUE_E[tier]}) ===")
    L = ensure_model(slug)
    restore(work)
    retire_stale(slug, work)
    want_engine = job_d.ENGINE_VERSION if causal else job_e.ENGINE_VERSION
    checks = json.loads(chk.read_text()) if restore(chk) else None
    if (not checks or not checks.get("pass") or checks.get("revision") != L["meta"]["revision"]
            or checks.get("mlm_reference_sha") != job_e.MLM_REFERENCE_SHA256 or checks.get("engine_version") != want_engine):
        checks = run_checks(slug, set_name, L, work)
        save_json(chk, checks)
    show(checks)
    if not checks["pass"]:
        msg = f"{slug}:{set_name}: a blocking check FAILED. Send {chk.name} back; do not use this model's numbers."
        if STOP_ON_FAIL:
            raise SystemExit(msg)
        print(msg)
        return
    hours = checks["timing"]["projected_hours"]
    finish = datetime.now(timezone.utc) + timedelta(hours=hours)
    late = finish.date().isoformat() > job_e.TIER_DUE_E[tier]
    print(f"  TIMING: first {checks['timing']['n_sentences']} sentences in {checks['timing']['seconds']} s -> projected "
          f"{hours:.1f} h for the set, finishing about {finish:%a %d %b %H:%M} UTC"
          + ("   !! later than this tier's due date -- tell the requesters now" if late else ""))
    if PAUSE_AFTER_TIMING:
        raise SystemExit(f"Paused after timing ({chk.name} is on Drive). Set PAUSE_AFTER_TIMING = False to continue.")
    started = time.perf_counter()
    rows = run_item(slug, set_name, L, work, sents, limit=LIMIT or None)
    print(f"  {len(rows)} / {len(sents)} sentences ({(time.perf_counter() - started) / 3600:.2f} h this session, "
          f"{tree_runner.wall_seconds(rows) / 3600:.2f} h of compute in all)")
    if LIMIT:
        print("  LIMIT > 0: dry run, no deliverable.")
        return
    final = dict(checks)
    if causal:
        final["causality_full"] = job_d.check_causality(rows, "float32")
        final["complete"] = job_d.check_complete(rows, sents)
    else:
        final["complete"] = job_e.check_complete_mlm(rows, sents)
    if not final["complete"]["pass"]:
        save_json(chk, final)
        raise SystemExit(f"{slug}:{set_name}: incomplete or non-finite rows: {final['complete']}")
    n_spans = sum(len(s["keys"]) for s in sents)
    if causal:
        meta = job_d.item_meta(job_e.GPT2_MEDIUM, set_name, L["meta"], n_sentences=len(sents), n_spans=n_spans,
                               wall_seconds=tree_runner.wall_seconds(rows), checks=final, budget=L["budget"],
                               extra={"calibration": L["calib"], "notebook": "experiment_job_e.ipynb"})
    else:
        meta = job_e.item_meta_e(job_e.MODELS_E[slug], set_name, L["meta"], n_sentences=len(sents), n_spans=n_spans,
                                 wall_seconds=tree_runner.wall_seconds(rows), checks=final, budget=L["budget"],
                                 extra={"calibration": L["calib"]})
    path = job_d.finalize_item(rows, sents, out, meta)
    save_json(chk, final)
    push(path)
    print(f"  wrote {path.name} ({path.stat().st_size / 2**20:.0f} MB)")


for slug, set_name, tier in QUEUE:
    run_one(slug, set_name, tier)
print("\nQueue finished.")
status()

## 9 · Tests (optional)

In [ ]:
#@title Run job E's tests inside Colab
import subprocess, sys

result_proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_job_e.py"], capture_output=True, text=True)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `A reference file differs from the package's` | `git checkout src/m1_analyzer/experiments/mlm_runner_ref.py src/m1_analyzer/experiments/tree_runner_ref_v4.py`. |
| Tokenizer check FAIL | Stop. Send the `mlmtok_*.json` / `tokenizer_*.json` files. The counts must equal the note's table; don't work around a mismatch. |
| `engine_equivalence` or `anchor` FAIL | Stop. Send `checks_<model>_<set>.json` back. |
| `!! later than this tier's due date` | Send the projected hours to the requesters. The note asks to be told before precision is changed or anything is cut. |
| Out of memory | Handled automatically: the batch is halved and the budget shrinks (`oom_backoffs` in the meta). |
| `… renamed to ….stale-…` | A working cache from another reference or engine version; the item restarts. |
| Colab disconnected | Re-run all cells. Finished items are skipped and the current one resumes. |